# EcoSentinel: Fine-Tuning TimesFM 2.5 with LoRA
## Does Google's forecasting model beat our LSTM once it studies our cities?

**What this notebook does, top to bottom:**
1. Clones the EcoSentinel project and installs `transformers`, `peft`, `accelerate`
2. Downloads 2024 PM2.5 for Delhi, Los Angeles, Seoul, Krakow and builds 32-day
   flashcards on **the same study / practice / exam days as the LSTM**
   (safety check: the 140 exam days and answers must match the saved test set)
3. Loads TimesFM 2.5 and scores it **before studying** (zero-shot, 32 days)
4. Freezes TimesFM and attaches small **LoRA** adapters (only ~0.6% of weights train)
5. Trains on the study days, keeping the version with the best **practice** score
6. Scores the fine-tuned model on the 140 exam days next to every other forecaster
7. Saves and downloads the results and the small LoRA adapter

**Why 32 days of history, not 30:** TimesFM reads history in 32-point patches. With
shorter inputs the Hugging Face model pads with zeros and includes them in its internal
mean and standard deviation, which distorts fine-tuning. The **exam days and answers
are unchanged**; TimesFM simply sees 2 extra past days. "Before vs after LoRA" in this
notebook both use 32 days, so that comparison is exact.

**Runtime:** Runtime → Change runtime type → **T4 GPU** recommended (CPU also works,
just slower).

In [ ]:
# ── Cell 1: Clone the repo and install the fine-tuning tools ─────────────────
# Takes ~1 to 2 minutes. If you see 'already exists', restart the runtime instead
# (Runtime → Disconnect and delete runtime) so you get the latest code.

!git clone https://github.com/maahipatel05/ecosentinel.git
%cd ecosentinel

!pip install transformers peft accelerate --quiet

print("\n✅ Installation complete")

---
## Step 1: Build the 32-day flashcards

Each flashcard: **32 days of PM2.5** (front) → **the next day's PM2.5** (back).

Piles are assigned by the **answer date**, using exactly the LSTM's days:

| Pile | Answer dates (per city) | Used for |
|---|---|---|
| Study | early Feb to late Oct 2024 | training the LoRA adapters |
| Practice | late Oct to Nov 26 | choosing the best version (no learning) |
| Exam | **Nov 27 to Dec 31** (35 days) | graded once, at the end |

Study has 2 fewer cards per city than the LSTM (266 vs 268) because each card needs
32 days of history. Hourly data is averaged to daily with the same shared
`hourly_to_daily` function the LSTM and the live `get_forecast` tool use.

In [ ]:
# ── Cell 2: Download 2024 PM2.5 and build the flashcards ─────────────────────
# Takes ~10 seconds.

import asyncio
import copy
import json
import math
import random
import sys
import time
from datetime import datetime, timezone
from pathlib import Path

import httpx
import numpy as np
import torch

sys.path.insert(0, "src")
from forecast import hourly_to_daily  # shared with the LSTM notebook and get_forecast

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

CONTEXT_LEN = 32
CITIES = {
    "Delhi":       (28.6517,   77.2219),
    "Los Angeles": (34.0522, -118.2437),
    "Seoul":       (37.5665,  126.9780),
    "Krakow":      (50.0647,   19.9450),
}
AIR_QUALITY_URL = "https://air-quality-api.open-meteo.com/v1/air-quality"
TEST_SET_PATH   = Path("data/eval_results/forecast_test_set.npz")
RESULTS_PATH    = Path("data/eval_results/timesfm_lora_results.json")
PREDS_PATH      = Path("data/eval_results/timesfm_lora_preds.npz")
ADAPTER_DIR     = Path("timesfm_lora_adapter")

# LSTM split, by answer-day index within each city's 366 days:
# study 30..297, practice 298..330, exam 331..365 (the LSTM's 30-day windows).
# With 32 days of history the first possible answer day is index 32.
STUDY_IDX    = range(CONTEXT_LEN, 298)
PRACTICE_IDX = range(298, 331)
EXAM_IDX     = range(331, 366)


async def fetch_pm25(lat, lon):
    async with httpx.AsyncClient(timeout=60) as client:
        r = await client.get(AIR_QUALITY_URL, params={
            "latitude": lat, "longitude": lon, "hourly": "pm2_5",
            "start_date": "2024-01-01", "end_date": "2024-12-31", "timezone": "UTC",
        })
    r.raise_for_status()
    h = r.json()["hourly"]
    daily = hourly_to_daily(h["time"], h["pm2_5"])
    dates = sorted(daily)
    return dates, np.array([daily[d] for d in dates], dtype=np.float32)


def make_cards(series, dates, idx_range):
    ctx = np.stack([series[t - CONTEXT_LEN : t] for t in idx_range])
    tgt = np.array([series[t] for t in idx_range], dtype=np.float32)
    return ctx, tgt, [dates[t] for t in idx_range]


piles = {name: {"ctx": [], "tgt": [], "city": [], "date": []} for name in ("study", "practice", "exam")}
for city, (lat, lon) in CITIES.items():
    dates, pm25 = await fetch_pm25(lat, lon)
    assert len(dates) == 366, f"{city}: expected 366 days, got {len(dates)}"
    for name, idx in (("study", STUDY_IDX), ("practice", PRACTICE_IDX), ("exam", EXAM_IDX)):
        c, t, d = make_cards(pm25, dates, idx)
        piles[name]["ctx"].append(c); piles[name]["tgt"].append(t)
        piles[name]["city"] += [city] * len(t); piles[name]["date"] += d
    print(f"  {city:<12} study {len(STUDY_IDX)} | practice {len(PRACTICE_IDX)} | exam {len(EXAM_IDX)}")

for name in piles:
    piles[name]["ctx"] = np.concatenate(piles[name]["ctx"])
    piles[name]["tgt"] = np.concatenate(piles[name]["tgt"])
    piles[name]["city"] = np.array(piles[name]["city"])
    piles[name]["date"] = np.array(piles[name]["date"])

# Safety check: the exam must be the identical 140 days and answers as the LSTM's test set
exam_saved = np.load(TEST_SET_PATH)
assert (piles["exam"]["date"] == exam_saved["date_te"]).all(), "exam dates differ from saved test set"
assert (piles["exam"]["city"] == exam_saved["city_te"]).all(), "exam cities differ from saved test set"
assert np.allclose(piles["exam"]["tgt"], exam_saved["y_te"], atol=1e-3), "exam answers differ"
assert np.allclose(piles["exam"]["ctx"][:, -30:], exam_saved["X_te"][:, :, 0], atol=1e-3), "last 30 days differ"

print(f"\nTotal: study {len(piles['study']['tgt'])} | practice {len(piles['practice']['tgt'])} | exam {len(piles['exam']['tgt'])}")
print("✅ Exam matches the saved test set (same 140 days, same answers, same last 30 days)")

---
## Step 2: Load TimesFM 2.5 and take the "before" photo

We load the **Hugging Face Transformers** version of TimesFM 2.5
(`google/timesfm-2.5-200m-transformers`, Apache-2.0) and score it on the exam
**before any training** (zero-shot), with the same 32-day history the fine-tuned model
will use. This is the exact "before" for "before vs after LoRA".

In [ ]:
# ── Cell 3: Load TimesFM 2.5 and score it zero-shot (32 days) ────────────────
# First run downloads ~1 GB (a few minutes on CPU runtimes, seconds on GPU).

from transformers import TimesFm2_5ModelForPrediction

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

model = TimesFm2_5ModelForPrediction.from_pretrained(
    "google/timesfm-2.5-200m-transformers",
    torch_dtype=torch.float32,  # full precision: the T4 GPU has no native bfloat16
).to(device)


def compute_metrics(y_true, y_pred, label):
    y_true = np.asarray(y_true, dtype=np.float32)
    y_pred = np.clip(np.asarray(y_pred, dtype=np.float32), 0, None)
    mae  = float(np.mean(np.abs(y_true - y_pred)))
    rmse = float(np.sqrt(np.mean((y_true - y_pred) ** 2)))
    mape = float(np.mean(np.abs((y_true - y_pred) / (y_true + 1e-8))) * 100)
    return {"model": label, "mae": round(mae, 3), "rmse": round(rmse, 3), "mape": round(mape, 2)}


@torch.no_grad()
def predict(m, ctx, batch_size=64):
    """Next-day point forecast + 10th/90th percentiles for each 32-day context."""
    m.eval()
    point, q10, q90 = [], [], []
    for i in range(0, len(ctx), batch_size):
        x = torch.tensor(ctx[i : i + batch_size], dtype=torch.float32, device=device)
        out = m(past_values=x, forecast_context_len=CONTEXT_LEN)
        point.append(out.mean_predictions[:, 0].float().cpu().numpy())
        q10.append(out.full_predictions[:, 0, 1].float().cpu().numpy())   # 10th percentile
        q90.append(out.full_predictions[:, 0, 9].float().cpu().numpy())   # 90th percentile
    return np.concatenate(point), np.concatenate(q10), np.concatenate(q90)


y_exam = piles["exam"]["tgt"]
t0 = time.time()
zs_pred, zs_q10, zs_q90 = predict(model, piles["exam"]["ctx"])
zs = compute_metrics(y_exam, zs_pred, "TimesFM 2.5 zero-shot (32 days)")
n_params = sum(p.numel() for p in model.parameters())
print(f"Loaded {n_params / 1e6:.0f}M parameters | forecast 140 days in {time.time() - t0:.1f}s")
print(f"\nBEFORE studying (zero-shot, 32 days):  MAE {zs['mae']:.2f}  RMSE {zs['rmse']:.2f}  MAPE {zs['mape']:.1f}%")

---
## Step 3: Freeze TimesFM and attach the LoRA adapters

**LoRA** (Low-Rank Adaptation) freezes every original weight and adds two small
trainable matrices, **B × A**, next to each linear layer. Settings follow Google's
official TimesFM 2.5 fine-tuning example:

| Setting | Value | Meaning |
|---|---|---|
| `r` | 4 | rank: how "thick" each adapter is |
| `lora_alpha` | 8 | scaling: the update counts ×(8 / 4) = ×2 |
| `lora_dropout` | 0.05 | randomly drops 5% of the adapter path while training |
| `target_modules` | `"all-linear"` | an adapter on every linear layer |

B starts at zero, so right after this cell the model still predicts exactly like
zero-shot TimesFM.

In [ ]:
# ── Cell 4: Attach LoRA (peft) ───────────────────────────────────────────────
# Takes a few seconds.

from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=4,
    lora_alpha=8,
    lora_dropout=0.05,
    target_modules="all-linear",
    bias="none",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())

---
## Step 4: Train (only the adapters learn)

Each epoch = one pass through the study cards in batches of 32:
1. TimesFM predicts each card's next day; **the model computes its own loss**
   (squared error on the forecast + a quantile loss for the ranges, in normalized units)
2. Only the LoRA adapters are nudged (AdamW, learning rate 1e-4, cosine schedule,
   gradient clipping at 1.0)
3. After each epoch we score the **practice** cards in real µg/m³ (MAE).
   **The best practice score is kept**, which protects against memorizing
   (we have ~1,060 study cards, far fewer than Google's example uses)

In [ ]:
# ── Cell 5: Fine-tune with LoRA, keeping the best practice score ─────────────
# ~1 to 3 minutes on a T4 GPU (longer on CPU).

EPOCHS, BATCH_SIZE, LR = 10, 32, 1e-4

study_ctx = torch.tensor(piles["study"]["ctx"], dtype=torch.float32)
study_tgt = torch.tensor(piles["study"]["tgt"], dtype=torch.float32).unsqueeze(1)  # (N, 1) = 1 day ahead
n_batches = math.ceil(len(study_ctx) / BATCH_SIZE)

optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=LR, weight_decay=0.01)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS * n_batches)
gen = torch.Generator().manual_seed(SEED)

practice_mae_before = compute_metrics(piles["practice"]["tgt"], predict(model, piles["practice"]["ctx"])[0], "")["mae"]
best_mae, best_epoch, best_state = practice_mae_before, 0, None
history = []
print(f"Practice MAE before training: {practice_mae_before:.2f}\n")
print(f"{'Epoch':>5}  {'Train loss':>10}  {'Practice MAE':>12}  Status")
print("-" * 48)

t0 = time.time()
for epoch in range(1, EPOCHS + 1):
    model.train()
    order = torch.randperm(len(study_ctx), generator=gen)
    losses = []
    for b in range(n_batches):
        idx = order[b * BATCH_SIZE : (b + 1) * BATCH_SIZE]
        out = model(
            past_values=study_ctx[idx].to(device),
            future_values=study_tgt[idx].to(device),
            forecast_context_len=CONTEXT_LEN,
        )
        out.loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        optimizer.zero_grad()
        scheduler.step()
        losses.append(out.loss.item())

    practice_mae = compute_metrics(piles["practice"]["tgt"], predict(model, piles["practice"]["ctx"])[0], "")["mae"]
    status = ""
    if practice_mae < best_mae:
        best_mae, best_epoch = practice_mae, epoch
        best_state = {k: v.detach().clone() for k, v in model.state_dict().items() if "lora_" in k}
        status = "✓ best so far, saved"
    history.append({"epoch": epoch, "train_loss": round(float(np.mean(losses)), 4), "practice_mae": round(practice_mae, 3)})
    print(f"{epoch:>5}  {np.mean(losses):>10.4f}  {practice_mae:>12.2f}  {status}")

if best_state is None:
    print("\nNo epoch beat the untrained model on practice days; keeping zero-shot behaviour.")
    for k, v in model.state_dict().items():
        if "lora_B" in k:
            v.zero_()
else:
    model.load_state_dict(best_state, strict=False)
    print(f"\n✅ Restored the best adapters from epoch {best_epoch} (practice MAE {best_mae:.2f}, "
          f"was {practice_mae_before:.2f} before training) in {time.time() - t0:.0f}s")

---
## Step 5: The final exam

The fine-tuned model predicts the same 140 exam days. Everyone is scored together:
baselines and the LSTM (from the saved test set, 30-day history), TimesFM zero-shot
(32 days), TimesFM + LoRA (32 days), and the Step 2 ensemble.

In [ ]:
# ── Cell 6: Score everyone on the exam and save ──────────────────────────────
# Takes a few seconds.

ft_pred, ft_q10, ft_q90 = predict(model, piles["exam"]["ctx"])
city_exam = piles["exam"]["city"]

predictions = {
    "Persistence (same as today)":     exam_saved["X_te"][:, -1, 0],
    "7-day rolling average":           exam_saved["X_te"][:, -7:, 0].mean(axis=1),
    "LSTM (ours, 30 days)":            exam_saved["preds_lstm"],
    "TimesFM 2.5 zero-shot (32 days)": zs_pred,
    "TimesFM 2.5 + LoRA (32 days)":    ft_pred,
}
ensemble_saved = json.loads(Path("data/eval_results/ensemble_results.json").read_text())


def print_table(rows, title):
    print(f"\n{title}")
    print(f"{'Model':<36}  {'MAE':>8}  {'RMSE':>8}  {'MAPE':>8}")
    print("-" * 66)
    for r in rows:
        print(f"{r['model']:<36}  {r['mae']:>8.2f}  {r['rmse']:>8.2f}  {r['mape']:>7.1f}%")


overall = [compute_metrics(y_exam, p, n) for n, p in predictions.items()]
overall.append({**next(r for r in ensemble_saved["overall"] if r["model"].startswith("Ensemble")),
                "model": "Ensemble LSTM + TimesFM zero-shot"})
print_table(overall, f"OVERALL ({len(y_exam)} exam days)")

per_city = {}
for city in CITIES:
    m = city_exam == city
    per_city[city] = [compute_metrics(y_exam[m], p[m], n) for n, p in predictions.items()]
    per_city[city].append({**next(r for r in ensemble_saved["per_city"][city] if r["model"].startswith("Ensemble")),
                           "model": "Ensemble LSTM + TimesFM zero-shot"})
    print_table(per_city[city], f"{city} ({int(m.sum())} exam days)")

coverage = {}
inside = (y_exam >= ft_q10) & (y_exam <= ft_q90)
for city in CITIES:
    coverage[city] = round(float(inside[city_exam == city].mean() * 100), 1)
coverage["overall"] = round(float(inside.mean() * 100), 1)
print("\nTimesFM + LoRA 80% interval coverage (ideal ≈ 80%):")
for k, v in coverage.items():
    print(f"  {k:<12} {v:5.1f}%")

ft = next(r for r in overall if r["model"].startswith("TimesFM 2.5 + LoRA"))
lstm = next(r for r in overall if r["model"].startswith("LSTM"))
gain = (zs["mae"] - ft["mae"]) / zs["mae"] * 100
vs_lstm = (lstm["mae"] - ft["mae"]) / lstm["mae"] * 100
print(f"\n→ LoRA vs zero-shot (both 32 days): MAE {zs['mae']:.2f} → {ft['mae']:.2f} "
      f"({abs(gain):.1f}% {'lower' if gain > 0 else 'higher'})")
print(f"→ LoRA vs LSTM: {ft['mae']:.2f} vs {lstm['mae']:.2f} ({abs(vs_lstm):.1f}% {'lower' if vs_lstm > 0 else 'higher'})")

results = {
    "run_timestamp": datetime.now(timezone.utc).isoformat(),
    "config": {
        "base_model": "google/timesfm-2.5-200m-transformers", "context_days": CONTEXT_LEN, "horizon": 1,
        "lora": {"r": 4, "alpha": 8, "dropout": 0.05, "target_modules": "all-linear"},
        "trainable_params": int(trainable), "total_params": int(total),
        "epochs": EPOCHS, "batch_size": BATCH_SIZE, "lr": LR, "optimizer": "AdamW (wd 0.01), cosine, clip 1.0",
        "seed": SEED, "device": device,
        "study_cards": int(len(piles["study"]["tgt"])), "practice_cards": int(len(piles["practice"]["tgt"])),
        "exam_days": int(len(y_exam)), "best_epoch": best_epoch,
        "practice_mae_before": round(practice_mae_before, 3), "practice_mae_best": round(best_mae, 3),
        "note": "LSTM and baselines use 30-day history; TimesFM uses 32 (same exam days and answers)",
    },
    "training_history": history,
    "overall": overall,
    "per_city": per_city,
    "lora_80pct_interval_coverage": coverage,
}
RESULTS_PATH.write_text(json.dumps(results, indent=2))
np.savez(PREDS_PATH, zs_pred=zs_pred, ft_pred=ft_pred, ft_q10=ft_q10, ft_q90=ft_q90,
         city_te=city_exam, date_te=piles["exam"]["date"])
model.save_pretrained(ADAPTER_DIR)
print(f"\n✅ Saved scores → {RESULTS_PATH}")
print(f"✅ Saved predictions → {PREDS_PATH}")
print(f"✅ Saved LoRA adapter → {ADAPTER_DIR}/")

In [ ]:
# ── Cell 7: Download the results and the adapter ─────────────────────────────
# Move the .json and .npz into ecosentinel/data/eval_results/.
# The adapter zip is the trained LoRA "sticky notes" (a few MB).

import shutil
from google.colab import files

shutil.make_archive("timesfm_lora_adapter", "zip", ADAPTER_DIR)
for path in (RESULTS_PATH, PREDS_PATH, Path("timesfm_lora_adapter.zip")):
    files.download(str(path))
print("✅ Files downloading, check your browser's downloads folder.")